In [1]:
!pip install ollama

     |████████████████████████████████| 472 kB 532 kB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 11.0 MB/s eta 0:00:01
You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
import ollama
import pandas as pd

MODEL = "apertus:8b-instruct-q4"
PROMPT = ("Übersetze den folgenden Schweizerdeutschen Satz ins Hochdeutsche. "
          "Behalte Bedeutung, Ton und alle Beleidigungen oder vulgären Ausdrücke wörtlich bei, "
          "schwäche nichts ab. Gib nur die Übersetzung aus.\n\nSatz: {text}")


def normalize(text):
    r = ollama.generate(model=MODEL, prompt=PROMPT.format(text=text), options={"temperature": 0})
    return r["response"].strip()

test50 = pd.read_csv("../data/annotation_dario.csv").head(50)
test50["apertus_q4"] = test50["text"].map(normalize)
test50.to_csv("../data/normalization_test50_apertus_q4.csv", index=False)

## Approach 2: translation with Claude Haiku 5.5

The final 700-sentence SwissCrawl sample is translated to Standard German with `claude-haiku-5-5` at effort `medium`. The production pipeline sends the approved numbered few-shot prompt in one batch request, validates all 700 numbered outputs, and stores them with the stratified master sample. The cells below document the earlier 50-sentence translation comparison, not the final corpus-generation run.

The client reads `ANTHROPIC_API_KEY` from the environment or `.env`.

In [1]:
from concurrent.futures import ThreadPoolExecutor

import anthropic
import pandas as pd
from dotenv import load_dotenv
from sacrebleu.metrics import BLEU, CHRF

load_dotenv()

HAIKU_MODEL = "claude-haiku-5-5"
EFFORT = "medium"
INPUT_PRICE_PER_TOKEN = 0.10 / 1e6
OUTPUT_PRICE_PER_TOKEN = 0.50 / 1e6
PARALLEL_REQUESTS = 8
INSTRUCTION = ("Übersetze den folgenden Schweizerdeutschen Satz ins Hochdeutsche. "
               "Behalte Bedeutung, Ton und alle Beleidigungen oder vulgären Ausdrücke wörtlich bei, "
               "schwäche nichts ab. Gib nur die Übersetzung aus.")
EXAMPLES = [
    ("De Sportverein en Nüereschdoorf bietet während de Sommermönet Yogaüebige onter freiem Hemmel of de Wese ah.",
     "Der Sportverein in Nürensdorf bietet während der Sommermonate Yogaübungen unter freiem Himmel auf der Wiese an."),
    ("In Deitige isch es üblich, die monatliche Fixköste für de \"V-Pass bequem per Duuruftrag z regle.",
     "In Deitingen ist es üblich, die monatlichen Fixkosten für den ÖV-Pass bequem per Dauerauftrag zu regeln."),
    ("D Omleitig förs kantonale Schötzefescht füehrt öber Hofdere ond Horb tüüf is ländliche Gebiet.",
     "Die Umleitung für das kantonale Schützenfest führt über Hochdorf und Horw tief in das ländliche Gebiet."),
    ("Nach de Entgleisig vomne Dienstfahrzüg in Remismüli blibt d Strecki fürd Spuresicherig polizeilich gsperrt.",
     "Nach der Entgleisung eines Dienstfahrzeugs in Rämismühle bleibt die Strecke für die Spurensicherung polizeilich gesperrt."),
]

client = anthropic.Anthropic()


def few_shot_prompt(text):
    examples = "\n\n".join(f"Schweizerdeutsch: {ch}\nHochdeutsch: {de}" for ch, de in EXAMPLES)
    return f"{INSTRUCTION}\n\n{examples}\n\nSchweizerdeutsch: {text}\nHochdeutsch:"


def translate(text):
    response = client.messages.create(
        model=HAIKU_MODEL,
        max_tokens=16000,
        output_config={"effort": EFFORT},
        messages=[{"role": "user", "content": few_shot_prompt(text)}],
    )
    if response.stop_reason == "refusal":
        return None, response.usage
    return "".join(block.text for block in response.content if block.type == "text").strip(), response.usage


def translate_all(texts):
    with ThreadPoolExecutor(max_workers=PARALLEL_REQUESTS) as pool:
        results = list(pool.map(translate, texts))
    translations = [translation for translation, _ in results]
    input_tokens = sum(usage.input_tokens for _, usage in results)
    output_tokens = sum(usage.output_tokens for _, usage in results)
    thinking_tokens = sum(usage.output_tokens_details.thinking_tokens for _, usage in results)
    cost = input_tokens * INPUT_PRICE_PER_TOKEN + output_tokens * OUTPUT_PRICE_PER_TOKEN
    print(f"{len(texts)} sentences: {input_tokens} input tokens, {output_tokens} output tokens "
          f"({thinking_tokens} thinking), ${cost:.4f}, {translations.count(None)} refused")
    return translations

### Check against the i4ds references

The 50 i4ds sentences have Standard German references, so the API run can be scored the same way as the earlier candidates.

In [2]:
# Evaluation now uses only the selected SwissCrawl sample.

50 sentences: 34503 input tokens, 27177 output tokens (24141 thinking), $0.0170, 0 refused
nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0 | nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0


,model,chrF++,BLEU
4,claude_haiku_5_5_few_shot_effort_medium,84.9,67.9
7,claude_haiku_5_5_api_few_shot,84.9,64.1
5,claude_haiku_5_5_few_shot_effort_max,83.8,65.3
6,claude_haiku_5_5_zero_shot_effort_medium,78.0,55.5
1,apertus_8b_q4_few_shot,77.0,56.9
0,apertus_8b_q4_zero_shot,72.4,48.3
3,untranslated_source,38.7,1.8
2,i4ds_t5base_wiki2,38.4,2.4


### SwissCrawl sample

The 50 SwissCrawl sentences from the comparison, next to the earlier Haiku runs pasted from claude.ai.

In [3]:
swisscrawl = pd.read_csv("../data/translation_test50.csv")
swisscrawl["pred_claude_haiku_5_5_api_few_shot"] = translate_all(swisscrawl["text"])
swisscrawl.to_csv("../data/translation_test50.csv", index=False)
swisscrawl[["text", "pred_claude_haiku_5_5_few_shot", "pred_claude_haiku_5_5_api_few_shot"]]

50 sentences: 33951 input tokens, 18210 output tokens (15566 thinking), $0.0125, 0 refused


,text,pred_claude_haiku_5_5_few_shot,pred_claude_haiku_5_5_api_few_shot
0,Esch so richtig agnähm gsi de wenig Räge,"Es war richtig angenehm, dieser wenige Regen.","Es war richtig angenehm, so wenig Regen."
1,"Känne 4 vo de Querschleeger, will sie au in Ch...","Ich kenne 4 von den Querschlägern, weil sie au...","Ich kenne vier von den Querschlägern, weil sie..."
2,sportlich gitts i de schwiiz wohl kein klub wo...,Sportlich gibt es in der Schweiz wohl keinen K...,Sportlich gibt es in der Schweiz wohl keinen K...
3,"wär het entschiede, dass mer nid meh als ei pe...","Wer hat entschieden, dass man nicht mehr als e...","Wer hat entschieden, dass man nicht mehr als e..."
4,"s'schlächtischte pay tv wältwiit, e einzigi sc...","Das schlechteste Pay-TV weltweit, eine einzige...","Das schlechteste Pay-TV weltweit, eine einzige..."
5,"find ihn sympathischer als vorher, will er no ...","Finde ihn sympathischer als vorher, weil er me...","Ich finde ihn sympathischer als vorher, weil e..."
6,"und überhaupt das argument, das mer d ware wil...","Und überhaupt das Argument, dass man die Ware ...","Und überhaupt das Argument, dass man die Ware ..."
7,Er isch die nächst Saison uf Uzwil im B go chü...,Er wird nächste Saison in Uzwil im B-Team heru...,Er ist in der nächsten Saison in Uzwil in der ...
8,"Merci, ächt, isch ä super Wiehncahtsüberraschi...","Danke, echt, das war eine super Weihnachtsüber...","Danke, echt, das war eine super Weihnachtsüber..."
9,Aber wo mir dete uf dene Stüehl abghockt sind ...,"Aber als wir dort auf den Stühlen saßen, fing ...","Aber als wir dort auf diesen Stühlen saßen, fi..."
